# Module 11 — From PCA/UMAP/t‑SNE to Classifiers

**Goal:** take the dimensionality structure we explored with PCA/UMAP/t‑SNE and **build supervised classifiers** that map features → labels.  
We'll use scikit‑learn Pipelines to avoid data leakage and do clean model selection.

**You will:**  
1. Recreate PCA/UMAP/t‑SNE on a tabular dataset.  
2. Train three baseline classifiers: Logistic Regression, Random Forest, and SVM.  
3. Evaluate with stratified train/test, cross‑validation, ROC AUC, and confusion matrices.  
4. Do a small GridSearchCV and inspect feature importance / coefficients.

## 0) Colab setup

In [ ]:
# If running on Colab, install anything missing
try:
    import umap
except Exception:
    %pip -q install umap-learn scikit-learn matplotlib pandas numpy

## 1) Imports & utilities

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_wine

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2) Load a tabular dataset

In [ ]:
# We'll use the scikit-learn Wine dataset as a stand‑in for an 'omics-like' table
# (rows = samples, columns = features, labels = class).
data = load_wine(as_frame=True)
X = data.data
y = data.target
X.head()

### 2a) Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

## 3) Unsupervised recap: PCA, UMAP, t‑SNE

In [ ]:
# PCA
Xz = StandardScaler().fit_transform(X_train)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
PC = pca.fit_transform(Xz)

plt.figure()
plt.scatter(PC[:,0], PC[:,1])
plt.title("PCA (train set) — 2 PCs")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

print("Explained variance (PC1+PC2):", np.sum(pca.explained_variance_ratio_[:2]).round(3))

In [ ]:
# UMAP
um = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
UM = um.fit_transform(Xz)

plt.figure()
plt.scatter(UM[:,0], UM[:,1])
plt.title("UMAP (train set)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.show()

In [ ]:
# t-SNE
ts = TSNE(n_components=2, random_state=RANDOM_STATE, init="pca", learning_rate="auto", perplexity=30)
TS = ts.fit_transform(Xz)

plt.figure()
plt.scatter(TS[:,0], TS[:,1])
plt.title("t-SNE (train set)")
plt.xlabel("tSNE1"); plt.ylabel("tSNE2")
plt.show()

> **Note:** We don't train on PCA/UMAP/t‑SNE coordinates. For classification, feed the **original features** through a Pipeline that handles scaling and the estimator. This avoids data leakage and keeps evaluation honest.

## 4) Baseline classifiers with Pipelines

In [ ]:
models = {
    "LogReg": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500, multi_class="auto", random_state=RANDOM_STATE))]),
    "RF":     Pipeline([("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))]),
    "SVM":    Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)
    acc = accuracy_score(y_test, pred)
    try:
        auc = roc_auc_score(y_test, proba, multi_class="ovr")
    except Exception:
        auc = np.nan
    results[name] = {"accuracy": acc, "roc_auc_ovr": auc}

pd.DataFrame(results).T

### 4a) Confusion matrix (best model)

In [ ]:
best = max(results.items(), key=lambda kv: kv[1]["accuracy"])[0]
best_pipe = models[best]
pred = best_pipe.predict(X_test)

disp = ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=data.target_names)
fig, ax = plt.subplots()
disp.plot(ax=ax, colorbar=False)
plt.title(f"Confusion Matrix — {best}")
plt.show()

print(classification_report(y_test, pred, target_names=data.target_names))

## 5) Cross‑validation & small grid search

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    "LogReg": {"clf__C":[0.1,1,10]},
    "RF":     {"clf__n_estimators":[200,400], "clf__max_depth":[None,10]},
    "SVM":    {"clf__C":[0.5,1,2], "clf__gamma":["scale","auto"]},
}

tuned = {}
for name, pipe in models.items():
    grid = GridSearchCV(pipe, param_grid[name], cv=cv, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    tuned[name] = {"best_score": grid.best_score_, "best_params": grid.best_params_, "estimator": grid.best_estimator_}

pd.DataFrame({k: {"cv_best_acc": v["best_score"], **v["best_params"]} for k,v in tuned.items()}).T

### 5a) ROC curves (one-vs-rest) for tuned model

In [ ]:
from sklearn.preprocessing import label_binarize
from itertools import cycle

best_name = max(tuned.items(), key=lambda kv: kv[1]["best_score"])[0]
est = tuned[best_name]["estimator"].fit(X_train, y_train)
y_score = est.predict_proba(X_test)

y_bin = label_binarize(y_test, classes=np.unique(y_train))
fig = plt.figure()
RocCurveDisplay.from_predictions(y_bin.ravel(), y_score.ravel())
plt.title(f"ROC AUC (OvR) — {best_name}")
plt.show()

In [ ]:
# --- UMAP plot colored by category (train fit only) ---

import numpy as np
import matplotlib.pyplot as plt

# If umap wasn't imported earlier:
try:
    umap.UMAP
except NameError:
    import umap

# 1) Standardize using training data only (to avoid leakage)
scaler_vis = StandardScaler().fit(X_train)
Xtr_z = scaler_vis.transform(X_train)
Xte_z = scaler_vis.transform(X_test)

# 2) Fit UMAP on TRAIN, then transform TEST
um_vis = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
UM_tr = um_vis.fit_transform(Xtr_z)
UM_te = um_vis.transform(Xte_z)

# 3) Plot, color by class labels
classes = np.unique(y_train)
plt.figure(figsize=(6,5))

for c in classes:
    idx_tr = (y_train == c)
    plt.scatter(UM_tr[idx_tr,0], UM_tr[idx_tr,1],
                s=18, alpha=0.85, label=f"train: {c}")

# Optionally overlay the test set with hollow markers
for c in np.unique(y_test):
    idx_te = (y_test == c)
    plt.scatter(UM_te[idx_te,0], UM_te[idx_te,1],
                s=36, alpha=0.85, facecolors='none', edgecolors='k', linewidths=0.7,
                label=f"test: {c}")

plt.title("UMAP embedding colored by category\n(fit on train, transformed test)")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(loc="best", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## **What does this analysis tell us about wine?**

The dataset you just analyzed comes from a classic wine chemistry study.  
Each sample represents a wine made from one of three grape cultivars grown in the same region of Italy.  
For each wine, chemists measured several chemical features; things like alcohol content, malic acid, magnesium, total phenols, flavanoids, and color intensity.  
The question is simple:  
> *Can we tell which grape variety a wine came from based only on its chemical composition?*

---

### **Explore the evidence**
- What patterns did you see in the PCA or UMAP plots?  
  Did wines from the same cultivar cluster together even though the algorithm didn’t know their labels?
- Which features seemed to separate the clusters the most?
- How accurate were the classifiers (Logistic Regression, SVM, Random Forest) when predicting cultivar from chemistry?

---

### **Interpret what that means**
- If unsupervised methods already revealed clear clustering, what does that imply about the natural structure of the data?  
- Since the classifiers achieved igh accuracy, what does that tell us about how distinctive the chemical fingerprints of each grape type are?  
- Which chemical variables might be biologically or chemically responsible for those differences?

---

### **Take-home message**
The results show that ... [answer to initial question about wine -> grapes].  

